# Week 10: MLOps, Model Evaluation & Explainable AI
## Notebook 06: Model Probability Calibration: Platt Scaling & Isotonic Regression

### 1. Conceptual Intuition & Real-World Use Case
In high-stakes decision systems (medical diagnostics, autonomous driving, loan approval):
- A model predicting a 90% probability of cancer means that out of 100 patients with this score, **exactly 90 should have cancer**.
- However, modern deep neural networks and Gradient Boosted Trees are notoriously **uncalibrated**: they are frequently overconfident (predicting 99% probability on samples where accuracy is only 75%).

**Model Calibration** aligns predicted probabilities with true empirical frequencies without altering classification rank ordering!

---

### 2. Mathematical Foundations: Expected Calibration Error (ECE)
Partition predictions into $M$ equally-spaced probability bins $B_m \subset (0, 1]$:

$$\text{ECE} = \sum_{m=1}^M \frac{|B_m|}{N} \left| \text{acc}(B_m) - \text{conf}(B_m) \right|$$
Where:
- $\text{conf}(B_m) = \frac{1}{|B_m|} \sum_{i \in B_m} \hat{p}_i$ (average predicted confidence in bin $m$).
- $\text{acc}(B_m) = \frac{1}{|B_m|} \sum_{i \in B_m} y_i$ (true empirical accuracy in bin $m$).

#### Platt Scaling:
Fits a univariate logistic sigmoid on raw validation logits $z$:
$$P_{\text{calibrated}}(y=1 \mid z) = \frac{1}{1 + \exp(A z + B)}$$
Parameters $A$ and $B$ are optimized via maximum likelihood on a holdout calibration set.

In [ ]:
import numpy as np

np.random.seed(42)
N = 1000

# Step 1: Simulate Overconfident Uncalibrated Model Predictions
# Raw probabilities pushed toward extreme 0 and 1
y_true = np.random.binomial(1, 0.4, size=N)
raw_logits = np.random.randn(N) + 2.0 * (y_true - 0.5)

# Uncalibrated probabilities (Sigmoid with temperature 0.5 -> Overconfident!)
probs_uncalibrated = 1.0 / (1.0 + np.exp(-raw_logits / 0.5))

# Step 2: Compute Expected Calibration Error (ECE)
def compute_ece(probs, y_true, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    
    bin_details = []
    for i in range(n_bins):
        bin_mask = (probs > bin_boundaries[i]) & (probs <= bin_boundaries[i + 1])
        bin_count = np.sum(bin_mask)
        
        if bin_count > 0:
            bin_acc = np.mean(y_true[bin_mask])
            bin_conf = np.mean(probs[bin_mask])
            abs_diff = abs(bin_acc - bin_conf)
            ece += (bin_count / len(probs)) * abs_diff
            bin_details.append((bin_boundaries[i+1], bin_acc, bin_conf, abs_diff))
            
    return ece, bin_details

ece_uncal, _ = compute_ece(probs_uncalibrated, y_true)
print(f"Uncalibrated Model ECE: {ece_uncal * 100:.2f}% (High Miscalibration!)")

# Step 3: Implement Platt Scaling (Fit A and B via Logistic Regression on Logits)
from sklearn.linear_model import LogisticRegression

calibrator = LogisticRegression()
# Train on raw logits (holdout calibration set)
calibrator.fit(raw_logits.reshape(-1, 1), y_true)

# Calibrated probabilities
probs_calibrated = calibrator.predict_proba(raw_logits.reshape(-1, 1))[:, 1]

ece_cal, details = compute_ece(probs_calibrated, y_true)
print(f"Platt Calibrated Model ECE: {ece_cal * 100:.2f}% (Significant Calibration Improvement!)")
print(f"Calibration Error Reduction: {(ece_uncal - ece_cal) / ece_uncal * 100:.1f}%")

### 3. Step-by-Step Code Walkthrough
1. **Reliability Binning:** Predictions were grouped into 10 confidence bins, calculating the discrepancy $|\text{acc} - \text{conf}|$.
2. **Platt Parameter Fitting:** Logistic regression learned scaling parameters $A$ and $B$, shrinking overconfident probability extremes toward true empirical base rates.

---

### 4. Industry Applications & Trade-offs
- **Insurance Underwriting & Loan Pricing:** Accurate expected loss calculations ($E[\text{Loss}] = P(\text{Default}) \times \text{Exposure}$) strictly require calibrated probabilities.
- **Platt Scaling vs Isotonic Regression:** Platt scaling assumes a parametric sigmoid curve. Isotonic regression is non-parametric (stepwise monotonic) and better suited for non-sigmoid distortions when $>1000$ calibration samples are available.

---

### 5. Top Interview Questions & Answers
**Q1: Does probability calibration change a model's ROC-AUC score?**
> **Answer:** No. Monotonic calibration methods (like Platt Scaling and Isotonic Regression) preserve the rank ordering of predictions. Since ROC-AUC is purely a rank-order metric, the AUC remains completely unchanged even as calibration error drops.

**Q2: Why are modern deep neural networks poorly calibrated compared to older networks?**
> **Answer:** Guo et al. (2017) showed that modern architectures use depth, Batch Normalization, and weight decay that minimize Cross-Entropy loss by pushing logits to extreme values, sacrificing probability calibration for raw classification accuracy.